# Задание 4. Деревья решений на данных Adult Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

## Загрузка данных

In [2]:
# Загрузка данных из UCI Machine Learning Repository
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
                'marital-status', 'occupation', 'relationship', 'race', 'sex', 
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

# Загрузка обучающей выборки
data_train = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

print(f"Train data shape: {data_train.shape}")
print(f"Train data info:")
data_train.info()

Train data shape: (32561, 15)
Train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [3]:
data_train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# Загрузка тестовой выборки
data_test = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test',
    names=column_names,
    na_values=' ?',
    skipinitialspace=True,
    skiprows=1  # Пропускаем первую строку с комментарием
)

print(f"Test data shape: {data_test.shape}")
data_test.head()

Test data shape: (16281, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K.


## Предобработка данных

In [5]:
# Очистка целевой переменной от точек в тестовой выборке
data_test['income'] = data_test['income'].str.replace('.', '', regex=False)

# Проверка уникальных значений
print("Train income values:", data_train['income'].unique())
print("Test income values:", data_test['income'].unique())

Train income values: ['<=50K' '>50K']
Test income values: ['<=50K' '>50K']


In [6]:
# Удаление строк с пропущенными значениями
data_train_clean = data_train.dropna()
data_test_clean = data_test.dropna()

print(f"Train data after cleaning: {data_train_clean.shape}")
print(f"Test data after cleaning: {data_test_clean.shape}")

Train data after cleaning: (32561, 15)
Test data after cleaning: (16281, 15)


In [7]:
# Разделение на признаки и целевую переменную
X_train = data_train_clean.drop('income', axis=1)
y_train = (data_train_clean['income'] == '>50K').astype(int)

X_test = data_test_clean.drop('income', axis=1)
y_test = (data_test_clean['income'] == '>50K').astype(int)

print(f"Target distribution in train: {y_train.value_counts()}")
print(f"Target distribution in test: {y_test.value_counts()}")

Target distribution in train: income
0    24720
1     7841
Name: count, dtype: int64
Target distribution in test: income
0    12435
1     3846
Name: count, dtype: int64


In [8]:
# Преобразование категориальных признаков с помощью One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# Выравнивание столбцов в train и test
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"Encoded train shape: {X_train_encoded.shape}")
print(f"Encoded test shape: {X_test_encoded.shape}")

Encoded train shape: (32561, 100)
Encoded test shape: (16281, 100)


## Обучение базовой модели

In [9]:
# Обучение дерева решений с ограничением глубины
tree = DecisionTreeClassifier(max_depth=3, random_state=17)
tree.fit(X_train_encoded, y_train)

# Предсказания
tree_predictions = tree.predict(X_test_encoded)

# Оценка качества
accuracy = accuracy_score(y_test, tree_predictions)
print(f"Accuracy with max_depth=3: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, tree_predictions))

Accuracy with max_depth=3: 0.8448

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.95      0.90     12435
           1       0.76      0.51      0.61      3846

    accuracy                           0.84     16281
   macro avg       0.81      0.73      0.76     16281
weighted avg       0.84      0.84      0.83     16281



## Подбор гиперпараметров

In [10]:
# GridSearch для подбора оптимальной глубины дерева
tree_params = {'max_depth': range(2, 11)}

locally_best_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=17),
    tree_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

locally_best_tree.fit(X_train_encoded, y_train)

,estimator,DecisionTreeC...ndom_state=17)
,param_grid,"{'max_depth': range(2, 11)}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [11]:
print("Best params:", locally_best_tree.best_params_)
print("Best cross validation score:", locally_best_tree.best_score_)

Best params: {'max_depth': 9}
Best cross validation score: 0.856116334709149


In [12]:
# Обучение дерева с оптимальными параметрами
best_depth = locally_best_tree.best_params_['max_depth']
tuned_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=17)
tuned_tree.fit(X_train_encoded, y_train)

# Предсказания на тестовой выборке
tuned_tree_predictions = tuned_tree.predict(X_test_encoded)

# Оценка качества
tuned_accuracy = accuracy_score(y_test, tuned_tree_predictions)
print(f"Accuracy with best max_depth={best_depth}: {tuned_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, tuned_tree_predictions))

Accuracy with best max_depth=9: 0.8586

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     12435
           1       0.79      0.54      0.64      3846

    accuracy                           0.86     16281
   macro avg       0.83      0.75      0.78     16281
weighted avg       0.85      0.86      0.85     16281



## Анализ важности признаков

In [13]:
# Топ-10 самых важных признаков
feature_importances = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': tuned_tree.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 most important features:")
print(feature_importances.head(10))

Top 10 most important features:
                              feature  importance
30  marital-status_Married-civ-spouse    0.396181
2                       education-num    0.205537
3                        capital-gain    0.197901
4                        capital-loss    0.067888
0                                 age    0.049299
5                      hours-per-week    0.032822
38         occupation_Exec-managerial    0.008042
24                  education_HS-grad    0.007765
1                              fnlwgt    0.006633
44          occupation_Prof-specialty    0.005948
